# 01 — Data Audit and Split Verification

This notebook checks that the project data are in place, confirms the processed split files built by `scripts/data_prep.py`, audits label balance and metadata coverage, and saves a few milestone-ready figures.

**Expected inputs**
- `DATA/processed/isic2019/train.csv`
- `DATA/processed/isic2019/val.csv`
- `DATA/processed/isic2019/test.csv`
- `DATA/processed/milk10k/external_val.csv`
- `DATA/processed/class_weights.json`

**Outputs**
- split summary table
- class distribution figure
- metadata missingness table
- sample image grids

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

random.seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [ ]:
# Project paths
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name.lower() == "code" else cwd
DATA = ROOT / "DATA"
PROCESSED = DATA / "processed"

FIG_DIR = ROOT / "outputs" / "figures" / "01_data_audit"
TABLE_DIR = ROOT / "outputs" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

expected_paths = {
    "train_csv": PROCESSED / "isic2019" / "train.csv",
    "val_csv": PROCESSED / "isic2019" / "val.csv",
    "test_csv": PROCESSED / "isic2019" / "test.csv",
    "milk_csv": PROCESSED / "milk10k" / "external_val.csv",
    "class_weights": PROCESSED / "class_weights.json",
    "metadata_scaler": PROCESSED / "metadata_scaler.pkl",
    "metadata_ohe": PROCESSED / "metadata_ohe.pkl",
}

print(f"ROOT: {ROOT}")
print(f"DATA: {DATA}")
print("\nFile check:")
for name, path in expected_paths.items():
    print(f" - {name:16s} | exists={path.exists()} | {path}")

### Optional one-time preprocessing

Only run the next cell if the processed CSVs do **not** exist yet.  
If `DATA/processed/` is already populated, skip it.

In [ ]:
# Optional: build processed files once
# import subprocess, sys
# subprocess.run([sys.executable, str(ROOT / "scripts" / "data_prep.py")], check=True)

In [ ]:
train_df = pd.read_csv(expected_paths["train_csv"])
val_df = pd.read_csv(expected_paths["val_csv"])
test_df = pd.read_csv(expected_paths["test_csv"])
milk_df = pd.read_csv(expected_paths["milk_csv"])

with open(expected_paths["class_weights"], "r") as f:
    class_weights = json.load(f)

print("train shape:", train_df.shape)
print("val shape:  ", val_df.shape)
print("test shape: ", test_df.shape)
print("milk shape: ", milk_df.shape)

In [ ]:
# Quick look at columns and first rows
print("TRAIN COLUMNS")
print(train_df.columns.tolist())
display(train_df.head(3))

print("\nVAL COLUMNS")
print(val_df.columns.tolist())
display(val_df.head(3))

print("\nMILK COLUMNS")
print(milk_df.columns.tolist())
display(milk_df.head(3))

In [ ]:
def summarize_split(df, split_name):
    out = {
        "split": split_name,
        "rows": len(df),
        "n_classes": df["label"].nunique(),
        "malignant_rate": df["is_malignant"].mean() if "is_malignant" in df.columns else np.nan,
        "missing_age": df["age_approx"].isna().mean() if "age_approx" in df.columns else np.nan,
        "missing_sex": df["sex"].isna().mean() if "sex" in df.columns else np.nan,
        "missing_site": df["anatom_site_general"].isna().mean() if "anatom_site_general" in df.columns else np.nan,
    }
    return out

summary_df = pd.DataFrame([
    summarize_split(train_df, "isic_train"),
    summarize_split(val_df, "isic_val"),
    summarize_split(test_df, "isic_test"),
    summarize_split(milk_df, "milk_external_val"),
])

display(summary_df)
summary_df.to_csv(TABLE_DIR / "split_summary.csv", index=False)
print(f"Saved: {TABLE_DIR / 'split_summary.csv'}")

In [ ]:
# Label distribution table
split_frames = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
    "milk_external_val": milk_df,
}

label_counts = []
for split_name, df in split_frames.items():
    vc = df["label"].value_counts().sort_index()
    for label, count in vc.items():
        label_counts.append({"split": split_name, "label": label, "count": count})

label_counts_df = pd.DataFrame(label_counts)
display(label_counts_df.head(20))
label_counts_df.to_csv(TABLE_DIR / "label_counts_by_split.csv", index=False)

In [ ]:
# Class distribution plot
pivot_counts = (
    label_counts_df
    .pivot(index="label", columns="split", values="count")
    .fillna(0)
    .sort_index()
)

ax = pivot_counts.plot(kind="bar", figsize=(12, 6))
ax.set_title("Class distribution by split")
ax.set_xlabel("Label")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution_by_split.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Metadata coverage
metadata_cols = [c for c in ["age_approx", "sex", "anatom_site_general"] if c in train_df.columns]
missingness = pd.DataFrame({
    "train_missing_pct": train_df[metadata_cols].isna().mean().round(4),
    "val_missing_pct": val_df[metadata_cols].isna().mean().round(4),
    "test_missing_pct": test_df[metadata_cols].isna().mean().round(4),
    "milk_missing_pct": milk_df[metadata_cols].isna().mean().round(4),
})
display(missingness)
missingness.to_csv(TABLE_DIR / "metadata_missingness.csv")

In [ ]:
# Encoded metadata columns created by data_prep.py
encoded_cols = [
    c for c in train_df.columns
    if c.startswith("sex_") or c.startswith("anatom_site_general_")
]

print("Encoded metadata columns:")
print(encoded_cols)
print(f"n_encoded_features = {len(encoded_cols)}")

In [ ]:
# Image path checks
def path_exists(rel_path):
    if pd.isna(rel_path):
        return False
    return (ROOT / rel_path).exists()

train_df["path_exists"] = train_df["image_path"].apply(path_exists)
val_df["path_exists"] = val_df["image_path"].apply(path_exists)
test_df["path_exists"] = test_df["image_path"].apply(path_exists)

print("Missing image paths")
print("train:", (~train_df["path_exists"]).sum())
print("val:  ", (~val_df["path_exists"]).sum())
print("test: ", (~test_df["path_exists"]).sum())

In [ ]:
def show_sample_grid(df, split_name, samples_per_class=3, max_classes=8):
    labels = sorted(df["label"].unique())[:max_classes]
    n_rows = len(labels)
    n_cols = samples_per_class

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    if n_rows == 1:
        axes = np.array([axes])

    for row_idx, label in enumerate(labels):
        subset = df[df["label"] == label].sample(
            n=min(samples_per_class, (df["label"] == label).sum()),
            random_state=42
        )
        for col_idx in range(n_cols):
            ax = axes[row_idx, col_idx]
            ax.axis("off")
            if col_idx < len(subset):
                rel_path = subset.iloc[col_idx]["image_path"]
                img = Image.open(ROOT / rel_path).convert("RGB")
                ax.imshow(img)
                ax.set_title(f"{label}")
    plt.suptitle(f"{split_name}: sample images by class", y=1.02)
    plt.tight_layout()
    out_path = FIG_DIR / f"{split_name}_sample_grid.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

show_sample_grid(train_df, "isic_train", samples_per_class=3, max_classes=min(8, train_df['label'].nunique()))

In [ ]:
# Malignant vs benign snapshot
malignancy_summary = pd.DataFrame({
    "split": ["train", "val", "test", "milk_external_val"],
    "malignant_pct": [
        train_df["is_malignant"].mean(),
        val_df["is_malignant"].mean(),
        test_df["is_malignant"].mean(),
        milk_df["is_malignant"].mean(),
    ]
})

display(malignancy_summary)

ax = malignancy_summary.plot(x="split", y="malignant_pct", kind="bar", legend=False, figsize=(8, 4))
ax.set_title("Malignant share by split")
ax.set_ylabel("Proportion malignant")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "malignant_share_by_split.png", dpi=200, bbox_inches="tight")
plt.show()

## Notes for Milestone II

After this notebook runs cleanly, you should have:
- confirmed that the processed split files load correctly,
- verified class imbalance,
- confirmed metadata availability,
- and saved figures/tables that can feed the checkpoint writeup.

The next notebook, `02_baseline_modeling.ipynb`, uses these same processed CSVs for the first pretrained baseline.